# Phase 3.2 - Query, Retrieval, and RAG Observability

This notebook validates the retrieval and answer path end-to-end against the FastAPI server:

1. Check API health/status.
2. Ingest documents when needed (`/ingest`).
3. Run sample queries against `/query`.
4. Inspect top-k ranking quality and similarity scores.
5. Ask a question via `/answer` and inspect grounded response fields.

In [ ]:
from __future__ import annotations

import os
import time
from pathlib import Path

import pandas as pd
import requests

BASE_URL = os.getenv("RAG_API_BASE_URL", "http://127.0.0.1:8000")
TOP_K = int(os.getenv("RAG_TOP_K", "3"))
TIMEOUT = int(os.getenv("RAG_TIMEOUT_SECONDS", "30"))
QUICK_MODE = os.getenv("RAG_QUICK_MODE", "true").lower() in {"1", "true", "yes"}
AUTO_INGEST = os.getenv("RAG_AUTO_INGEST", "true").lower() in {"1", "true", "yes"}
INGEST_MAX_DOCS = int(os.getenv("RAG_INGEST_MAX_DOCS", "15"))
QUESTION_FOR_RAG = os.getenv("RAG_NOTEBOOK_QUESTION", "What is Uruguay?")

project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

print(f"Project root: {project_root}")
print(f"API base: {BASE_URL}, timeout: {TIMEOUT}s")
print(f"quick_mode={QUICK_MODE}, auto_ingest={AUTO_INGEST}, ingest_max_docs={INGEST_MAX_DOCS}")
print(f"top_k={TOP_K}, question_for_rag={QUESTION_FOR_RAG!r}")

In [ ]:
def get_json(path: str, params: dict | None = None) -> dict:
    response = requests.get(f"{BASE_URL}{path}", params=params, timeout=TIMEOUT)
    response.raise_for_status()
    return response.json()


def post_json(path: str, params: dict | None = None, payload: dict | None = None) -> dict:
    response = requests.post(f"{BASE_URL}{path}", params=params, json=payload, timeout=TIMEOUT)
    response.raise_for_status()
    return response.json()

In [ ]:
health = get_json("/health")
status = get_json("/status")

print("Health:", health)
print("Status:", status)

vector_loaded = status.get("vector_store_loaded", False)
if not vector_loaded:
    if AUTO_INGEST:
        print(f"No vectors loaded. Running ingest with max_docs={INGEST_MAX_DOCS}...")
        ingest_result = post_json("/ingest", params={"max_docs": INGEST_MAX_DOCS, "reingest": False})
        print("Ingest result:", ingest_result)
    else:
        print("No vectors loaded. AUTO_INGEST is false; run /ingest separately if needed.")

status = get_json("/status")
print("Updated status:", status)

In [ ]:
queries = [
    "What is Uruguay?",
    "Where is Montevideo located?",
    "How many people live in Uruguay?",
]

query_limit = 1 if QUICK_MODE else len(queries)
results = []

for query in queries[:query_limit]:
    t0 = time.perf_counter()
    response = post_json("/query", payload={"query": query, "k": TOP_K})
    dt_ms = (time.perf_counter() - t0) * 1000

    top_docs = response.get("top_docs", [])
    scores = response.get("scores", [])

    for rank, (doc, score) in enumerate(zip(top_docs, scores), start=1):
        doc_text = doc.get("text", "") if isinstance(doc, dict) else str(doc)
        results.append(
            {
                "query": query,
                "rank": rank,
                "score": float(score),
                "latency_ms": dt_ms,
                "doc_preview": doc_text.replace("\n", " ")[:140],
            }
        )

retrieval_df = pd.DataFrame(results)
retrieval_df.head(20)

In [ ]:
if retrieval_df.empty:
    print("No retrieval rows returned.")
else:
    display_cols = ["query", "rank", "score", "latency_ms", "doc_preview"]
    display(retrieval_df[display_cols].sort_values(["query", "rank"]).head(30))

    latency_df = retrieval_df.groupby("query", as_index=False)["latency_ms"].max().sort_values("latency_ms")
    print("\nLatency summary (ms):")
    print(latency_df.describe(include="all"))

In [ ]:
if not retrieval_df.empty:
    import matplotlib.pyplot as plt

    first_query = retrieval_df.iloc[0]["query"]
    first_df = retrieval_df[retrieval_df["query"] == first_query].sort_values("rank")

    plt.figure(figsize=(8, 4))
    plt.bar(first_df["rank"].astype(str), first_df["score"])
    plt.title(f"Top-{TOP_K} similarity scores for first query")
    plt.xlabel("Rank")
    plt.ylabel("Score")
    plt.tight_layout()
    plt.show()

    latency_df = retrieval_df.groupby("query", as_index=False)["latency_ms"].max().sort_values("latency_ms", ascending=False)
    plt.figure(figsize=(10, 4))
    plt.barh(latency_df["query"], latency_df["latency_ms"])
    plt.title("Query latency (ms)")
    plt.xlabel("Latency ms")
    plt.tight_layout()
    plt.show()

In [ ]:
answer_response = post_json("/answer", payload={"question": QUESTION_FOR_RAG, "k": TOP_K})

print("Question:", answer_response.get("question"))
print("Answer:", answer_response.get("answer"))
print("Retrieved docs:", len(answer_response.get("top_docs", [])))
print("Scores:", answer_response.get("scores", []))

answer_response